# **CHƯƠNG 6: NEURAL NETWORKS**

Notebook này hiện thực hóa các kiến thức nền tảng về Neural Networks được trình bày trong **Chương 6** trong cuốn sách *Speech and Language Processing (Jurafsky & Martin)*.

Bài thực hành này bao gồm:

1. **Giải quyết bài toán XOR (Section 6.2):** Chứng minh vì sao các mô hình tuyến tính (như Perceptron hay Logistic Regression) thất bại và sức mạnh của Lớp ẩn (Hidden Layer) cùng hàm phi tuyến (ReLU).

2. **Phân loại cảm xúc văn bản (Section 6.4 & 6.5):** Ứng dụng Feedforward Neural Network (FNN) kết hợp Word Embeddings và kỹ thuật Mean-pooling để phân tích cảm xúc (Sentiment Analysis).

3. **Phân loại cảm xúc cho từng khía cạnh trên bộ dữ liệu VLSP 2018 - Restaurant**

4. **Phân loại cảm xúc cho từng khía cạnh trên bộ dữ liệu VLSP 2018 - Hotel**

## **0. Cài đặt thư viện cần thiết**

In [ ]:
pip install torch matplotlib datasets

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import re
import os
from collections import Counter
from datasets import load_dataset # Dùng để tải dataset IMDB

print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.10.0+cpu


## **1. Bài toán XOR**

Như sách đã đề cập trong **Mục 6.2 (Hình 6.5)**, hàm XOR (Exclusive OR) là một hàm không thể phân tách tuyến tính (not linearly separable). Nghĩa là ta không thể vẽ một đường thẳng duy nhất để chia tách các điểm có kết quả `1` và `0`.

**Giải pháp (Hình 6.6):** Chúng ta sẽ xây dựng một mạng Feedforward gồm 2 lớp (2-layer network):
- **Input:** 2 features ($x_1, x_2$).
- **Hidden Layer:** 2 nodes ($h_1, h_2$) kết hợp hàm kích hoạt phi tuyến **ReLU** (Rectified Linear Unit).
- **Output Layer:** 1 node kết hợp hàm **Sigmoid** để đưa ra xác suất (từ 0 đến 1).

In [ ]:
# 1. Chuẩn bị dữ liệu XOR
X_xor = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

# 2. Xây dựng kiến trúc mạng (như Hình 6.6 trong sách)
class XORNetwork(nn.Module):
    def __init__(self):
        super(XORNetwork, self).__init__()
        # Lớp ẩn: 2 input -> 2 hidden nodes
        self.hidden = nn.Linear(2, 2)
        # Hàm kích hoạt ReLU (Công thức 6.6)
        self.relu = nn.ReLU()
        # Lớp đầu ra: 2 hidden -> 1 output node
        self.output = nn.Linear(2, 1)
        # Sigmoid map giá trị về khoảng [0, 1] (Công thức 6.3)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        h = self.hidden(x)
        a = self.relu(h)
        z = self.output(a)
        y_hat = self.sigmoid(z)
        return y_hat

# Khởi tạo mô hình
xor_model = XORNetwork()

# 3. Loss & Optimizer (Sử dụng Cross-Entropy Loss - Công thức 6.25)
criterion_xor = nn.BCELoss() # Binary Cross Entropy
optimizer_xor = optim.SGD(xor_model.parameters(), lr=0.1)

# 4. Huấn luyện (Training Loop với Backpropagation)
print("--- Training XOR Model ---")
for epoch in range(5000):
    optimizer_xor.zero_grad()      # Reset gradients
    preds = xor_model(X_xor)       # Forward pass
    loss = criterion_xor(preds, y_xor) # Tính Loss
    loss.backward()                # Backward pass (Tính đạo hàm)
    optimizer_xor.step()           # Cập nhật trọng số (w, b)

    if (epoch + 1) % 1000 == 0:
        print(f"Epoch {epoch+1:4d} | Loss: {loss.item():.4f}")

# Kiểm tra kết quả XOR
print("\n--- Kết quả XOR sau huấn luyện ---")
with torch.no_grad():
    for i in range(len(X_xor)):
        pred = xor_model(X_xor[i]).item()
        print(f"Input: {X_xor[i].tolist()} -> Target: {y_xor[i].item()} | Dự đoán: {pred:.4f}")

--- Training XOR Model ---
Epoch 1000 | Loss: 0.4799
Epoch 2000 | Loss: 0.4782
Epoch 3000 | Loss: 0.4778
Epoch 4000 | Loss: 0.4777
Epoch 5000 | Loss: 0.4776

--- Kết quả XOR sau huấn luyện ---
Input: [0.0, 0.0] -> Target: 0.0 | Dự đoán: 0.3337
Input: [0.0, 1.0] -> Target: 1.0 | Dự đoán: 0.3337
Input: [1.0, 0.0] -> Target: 1.0 | Dự đoán: 0.9991
Input: [1.0, 1.0] -> Target: 0.0 | Dự đoán: 0.3337


## **2. FNN cho Xử lý Ngôn ngữ Tự nhiên (NLP) - Phân tích Cảm xúc (Sentiment Analysis) sử dụng Word Embeddings**

- Bước sang **Mục 6.4 và 6.5**, sách chuyển hướng sang ứng dụng FNN vào xử lý ngôn ngữ. Thay vì dùng các đặc trưng thủ công (hand-built features), chúng ta sẽ biểu diễn từ vựng bằng **Word Embeddings** (Ma trận $E$).
- **Bộ dữ liệu (Dataset):** Bộ dữ liệu **IMDB Large Movie Review Dataset** (Maas et al., 2011) tại [Kaggle](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews). Tuy nhiên, để tiện lợi, ta sẽ tải trực tiếp qua thư viện `datasets` của Hugging Face.
- *Lưu ý:* Để quá trình demo diễn ra nhanh chóng, bài thực hành này sẽ trích xuất một mẫu nhỏ gồm 5000 câu để huấn luyện (train) và 1000 câu để kiểm thử (test)."

In [ ]:
# 1. Tải Dataset IMDB từ thư viện Hugging Face
dataset = load_dataset("imdb")

# Để demo chạy nhanh, ta chỉ lấy 5000 câu để train và 1000 câu để test
train_data = dataset['train'].shuffle(seed=42).select(range(5000))
test_data = dataset['test'].shuffle(seed=42).select(range(1000))

print(f"Số lượng câu Train: {len(train_data)}, Test: {len(test_data)}")
print(f"Ví dụ câu đầu tiên: {train_data[0]['text'][:100]}... | Nhãn: {train_data[0]['label']}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Số lượng câu Train: 5000, Test: 1000
Ví dụ câu đầu tiên: There is no relation at all between Fortier and Profiler but the fact that both are police series ab... | Nhãn: 1


In [ ]:
# 2. Tiền xử lý dữ liệu và Xây dựng Từ điển (Vocabulary)
def tokenize(text):
    # Tách từ đơn giản, loại bỏ dấu câu và chuyển chữ thường
    return re.findall(r'\w+', text.lower())

# Đếm tần suất từ
vocab_counter = Counter()
for item in train_data:
    vocab_counter.update(tokenize(item['text']))

# Giới hạn từ điển: lấy 10,000 từ phổ biến nhất
MAX_VOCAB_SIZE = 10000
vocab = {'<PAD>': 0, '<UNK>': 1} # Thêm token cho Padding và từ chưa biết (Unknown)
for word, count in vocab_counter.most_common(MAX_VOCAB_SIZE - 2):
    vocab[word] = len(vocab)

print(f"Kích thước từ điển: {len(vocab)} từ.")

Kích thước từ điển: 10000 từ.


In [ ]:
# Hàm mã hóa câu văn thành danh sách các index
MAX_SEQ_LEN = 150 # Độ dài tối đa của câu (N)

def encode_sentences(data):
    encoded = []
    labels = []
    for item in data:
        tokens = tokenize(item['text'])
        # Chuyển từ thành index, nếu không có trong vocab thì gán là <UNK>
        indices = [vocab.get(w, vocab['<UNK>']) for w in tokens[:MAX_SEQ_LEN]]
        # Padding nếu câu ngắn hơn MAX_SEQ_LEN
        if len(indices) < MAX_SEQ_LEN:
            indices += [vocab['<PAD>']] * (MAX_SEQ_LEN - len(indices))

        encoded.append(indices)
        labels.append(item['label'])
    return torch.tensor(encoded, dtype=torch.long), torch.tensor(labels, dtype=torch.long)

X_train, y_train = encode_sentences(train_data)
X_test, y_test = encode_sentences(test_data)

In [ ]:
# 3. Xây dựng Mô hình NLP với Mean-Pooling (Theo Hình 6.13)
class IMDBSentimentNet(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super(IMDBSentimentNet, self).__init__()
        # 1. Lớp Embedding E
        self.embedding = nn.Embedding(num_embeddings=vocab_size,
                                      embedding_dim=embed_dim,
                                      padding_idx=0)

        # 2. Các lớp Feedforward
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # x có shape: [batch_size, N]
        embeds = self.embedding(x) # -> Shape: [batch_size, N, d]

        # POOLING: Lấy trung bình cộng các vector từ trong câu (Công thức 6.21)
        x_mean = torch.mean(embeds, dim=1) # -> Shape: [batch_size, d]

        # Đưa qua mạng FNN (Công thức 6.22)
        h = self.fc1(x_mean)
        a = self.relu(h)
        z = self.fc2(a) # Output logits (z)

        return z

In [ ]:
# Khởi tạo mô hình
embed_dim = 64      # Chiều (d) của embedding vector
hidden_dim = 32     # Số node trong lớp ẩn (d_h)
num_classes = 2     # Tích cực (1) / Tiêu cực (0)

nlp_model = IMDBSentimentNet(len(vocab), embed_dim, hidden_dim, num_classes)

# Loss & Optimizer
# CrossEntropyLoss trong PyTorch đã tự động tính Softmax bên trong nó (Công thức 6.28)
criterion_nlp = nn.CrossEntropyLoss()
optimizer_nlp = optim.Adam(nlp_model.parameters(), lr=0.005)

In [ ]:
# 4. Huấn luyện mô hình NLP
epochs_nlp = 15
batch_size = 64

print("\n--- Bắt đầu huấn luyện mô hình IMDB Sentiment ---")
for epoch in range(epochs_nlp):
    nlp_model.train() # Chuyển mô hình sang chế độ train
    total_loss = 0
    correct = 0

    # Huấn luyện theo mini-batches
    for i in range(0, len(X_train), batch_size):
        batch_X = X_train[i:i+batch_size]
        batch_y = y_train[i:i+batch_size]

        optimizer_nlp.zero_grad()
        logits = nlp_model(batch_X)
        loss = criterion_nlp(logits, batch_y)

        loss.backward()
        optimizer_nlp.step()

        total_loss += loss.item()

        # Tính Accuracy
        preds = torch.argmax(logits, dim=1)
        correct += (preds == batch_y).sum().item()

    avg_loss = total_loss / (len(X_train) / batch_size)
    accuracy = correct / len(X_train) * 100
    print(f"Epoch {epoch+1:2d} | Train Loss: {avg_loss:.4f} | Train Acc: {accuracy:.2f}%")


--- Bắt đầu huấn luyện mô hình IMDB Sentiment ---
Epoch  1 | Train Loss: 0.6244 | Train Acc: 64.98%
Epoch  2 | Train Loss: 0.3679 | Train Acc: 83.50%
Epoch  3 | Train Loss: 0.2106 | Train Acc: 92.04%
Epoch  4 | Train Loss: 0.1183 | Train Acc: 96.42%
Epoch  5 | Train Loss: 0.0909 | Train Acc: 96.80%
Epoch  6 | Train Loss: 0.1140 | Train Acc: 95.24%
Epoch  7 | Train Loss: 0.0498 | Train Acc: 98.40%
Epoch  8 | Train Loss: 0.0300 | Train Acc: 99.06%
Epoch  9 | Train Loss: 0.0068 | Train Acc: 99.92%
Epoch 10 | Train Loss: 0.0047 | Train Acc: 99.94%
Epoch 11 | Train Loss: 0.0035 | Train Acc: 99.94%
Epoch 12 | Train Loss: 0.0024 | Train Acc: 99.98%
Epoch 13 | Train Loss: 0.0015 | Train Acc: 100.00%
Epoch 14 | Train Loss: 0.0011 | Train Acc: 100.00%
Epoch 15 | Train Loss: 0.0009 | Train Acc: 100.00%


In [ ]:
# 5. Đánh giá trên tập Test
nlp_model.eval() # Chế độ evaluation (tắt dropout nếu có)
with torch.no_grad():
    test_logits = nlp_model(X_test)
    test_preds = torch.argmax(test_logits, dim=1)
    test_correct = (test_preds == y_test).sum().item()
    test_acc = test_correct / len(X_test) * 100
    print(f"\n=> Độ chính xác trên tập Test: {test_acc:.2f}%")


=> Độ chính xác trên tập Test: 80.20%


In [ ]:
# Demo dự đoán một câu cụ thể:
sample_texts = [
    "This movie was fantastic and acting was great!",
    "What a terrible and boring film. I hated it."
]
sample_X, _ = encode_sentences([{'text': t, 'label': 0} for t in sample_texts])
with torch.no_grad():
    sample_logits = nlp_model(sample_X)
    sample_preds = torch.argmax(sample_logits, dim=1)

print("\n--- Demo thử nghiệm thực tế ---")
for text, pred in zip(sample_texts, sample_preds):
    label_str = "Tích cực (Positive)" if pred.item() == 1 else "Tiêu cực (Negative)"
    print(f"Câu: '{text}' \n--> Dự đoán: {label_str}\n")


--- Demo thử nghiệm thực tế ---
Câu: 'This movie was fantastic and acting was great!' 
--> Dự đoán: Tích cực (Positive)

Câu: 'What a terrible and boring film. I hated it.' 
--> Dự đoán: Tiêu cực (Negative)



## **3. Aspect-Based Sentiment Analysis với VLSP 2018 - Restaurant**

- Ở phần này, bài thực hành demo bài toán: Phân loại cảm xúc cho **từng khía cạnh** (Aspect) xuất hiện trong câu. Sau đó tổng hợp lại để đánh giá cảm xúc của toàn câu.
- **Bộ dữ liệu (Dataset)**: Bộ dữ liệu VLSP 2018 **(Aspect-Based Sentiment Analysis - Restaurant)**

In [ ]:
# 1. Đọc và trích xuất dữ liệu từ file txt VLSP 2018

polarity_map = {'negative': 0, 'neutral': 1, 'positive': 2}
inv_polarity_map = {0: 'NEGATIVE', 1: 'NEUTRAL', 2: 'POSITIVE'}

def parse_vlsp_absa(filepath):
    if not os.path.exists(filepath):
        print(f"[CẢNH BÁO] Không tìm thấy {filepath}.")
        return []

    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read().strip()

    blocks = re.split(r'\n\s*\n', content)
    parsed_data = []
    aspect_pattern = re.compile(r'\{([^,]+),\s*(positive|negative|neutral)\}')

    for block in blocks:
        lines = block.split('\n')
        if len(lines) >= 3:
            text = lines[1].strip()
            label_line = lines[2].strip()

            aspects = []
            for aspect, polarity in aspect_pattern.findall(label_line):
                aspects.append((aspect.strip(), polarity_map[polarity.strip()]))

            if len(aspects) > 0:
                parsed_data.append({'text': text, 'aspects': aspects})

    return parsed_data

# Tải dataset từ Google Drive
train_data = parse_vlsp_absa('1-VLSP2018-SA-Restaurant-train (7-3-2018).txt')
dev_data = parse_vlsp_absa('2-VLSP2018-SA-Restaurant-dev (7-3-2018).txt')
test_data = parse_vlsp_absa('3-VLSP2018-SA-Restaurant-test (8-3-2018).txt')

if train_data:
    print(f"\n--- Dữ liệu VLSP 2018 ---")
    print(f"Số lượng câu - Train: {len(train_data)}, Dev: {len(dev_data)}, Test: {len(test_data)}")
    print(f"Ví dụ câu 1:\n- Text: {train_data[0]['text']}\n- Aspect & Polarity: {train_data[0]['aspects']}")


--- Dữ liệu VLSP 2018 ---
Số lượng câu - Train: 2961, Dev: 1290, Test: 500
Ví dụ câu 1:
- Text: _ Ảnh chụp từ hôm qua, đi chơi với gia đình và 1 nhà họ hàng đang sống tại Sài Gòn. _ Hôm qua đi ăn trưa muộn, ai cũng đói hết nên lúc có đồ ăn là nhào vô ăn liền, bởi vậy mới quên chụp các phần gọi thêm với nước mắm, chỉ chụp món chính thôi! _ Đói quá nên không biết đánh giá đồ ăn kiểu gì luôn 😅😅😅_ Chọn cái này vì thấy nó lạ với tui.
- Aspect & Polarity: [('FOOD#STYLE&OPTIONS', 1), ('FOOD#QUALITY', 1)]


In [ ]:
# 2. Xây dựng từ điển (Vocabulary) cho từ và khía cạnh
def tokenize_vn(text):
    # Tách từ đơn giản bằng regex (như mô tả trong giáo trình cơ bản)
    return re.findall(r'\w+', text.lower())

word_vocab = {'<PAD>': 0, '<UNK>': 1}
aspect_vocab = {'<UNK>': 0}
word_counter = Counter()

# Đếm tần suất
for item in train_data:
    word_counter.update(tokenize_vn(item['text']))
    for aspect_name, _ in item['aspects']:
        if aspect_name not in aspect_vocab:
            aspect_vocab[aspect_name] = len(aspect_vocab)

# Giới hạn từ điển: 5000 từ phổ biến nhất
for word, _ in word_counter.most_common(5000):
    word_vocab[word] = len(word_vocab)

MAX_SEQ_LEN = 100

# Trải phẳng dữ liệu: 1 câu có N khía cạnh -> tạo ra N mẫu huấn luyện độc lập
def flatten_and_encode(data):
    X_texts, X_aspects, y_labels = [], [], []
    for item in data:
        tokens = tokenize_vn(item['text'])
        # Padding hoặc Cắt ngắn câu
        text_idx = [word_vocab.get(w, word_vocab['<UNK>']) for w in tokens[:MAX_SEQ_LEN]]
        text_idx += [word_vocab['<PAD>']] * max(0, MAX_SEQ_LEN - len(text_idx))

        for aspect_name, polarity in item['aspects']:
            aspect_idx = aspect_vocab.get(aspect_name, aspect_vocab['<UNK>'])
            X_texts.append(text_idx)
            X_aspects.append(aspect_idx)
            y_labels.append(polarity)

    return torch.tensor(X_texts, dtype=torch.long), \
           torch.tensor(X_aspects, dtype=torch.long), \
           torch.tensor(y_labels, dtype=torch.long)

X_train_txt, X_train_asp, y_train = flatten_and_encode(train_data)
X_dev_txt, X_dev_asp, y_dev = flatten_and_encode(dev_data)

In [ ]:
# 3. Xây dựng mô hình FEEDFORWARD NET cho ABSA (Dựa trên Hình 6.13)
class ABSAMeanPoolingNet(nn.Module):
    def __init__(self, vocab_size, aspect_size, word_dim, aspect_dim, hidden_dim, num_classes):
        super(ABSAMeanPoolingNet, self).__init__()
        # Embedding Layers (Section 6.5)
        self.word_emb = nn.Embedding(vocab_size, word_dim, padding_idx=0)
        self.aspect_emb = nn.Embedding(aspect_size, aspect_dim)

        # FNN Layers (Section 6.3)
        # Input của mạng ẩn là sự kết hợp (Concatenation) của Câu và Khía cạnh
        self.fc1 = nn.Linear(word_dim + aspect_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x_text, x_aspect):
        # 1. Mã hóa câu và Mean-pooling (Eq 6.21: x_mean = 1/N * sum(e_i))
        word_vectors = self.word_emb(x_text)
        sentence_rep = torch.mean(word_vectors, dim=1)

        # 2. Mã hóa khía cạnh
        aspect_rep = self.aspect_emb(x_aspect)

        # 3. Kết hợp (Concatenate)
        combined = torch.cat((sentence_rep, aspect_rep), dim=1)

        # 4. Truyền qua Feedforward Network (Eq 6.22)
        logits = self.fc2(self.relu(self.fc1(combined)))
        return logits

In [ ]:
# Khởi tạo mô hình
model = ABSAMeanPoolingNet(vocab_size=len(word_vocab),
                           aspect_size=len(aspect_vocab),
                           word_dim=128, aspect_dim=32,
                           hidden_dim=64, num_classes=3)

# Sử dụng CrossEntropyLoss (Eq 6.28) tích hợp sẵn Softmax
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.002)

In [ ]:
# 4. Huấn luyện mô hình
epochs = 50
batch_size = 64

if len(X_train_txt) > 0:
    print("\n--- BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH ABSA ---")
    for epoch in range(epochs):
        model.train()
        total_loss, correct = 0, 0

        for i in range(0, len(X_train_txt), batch_size):
            b_txt = X_train_txt[i:i+batch_size]
            b_asp = X_train_asp[i:i+batch_size]
            b_lbl = y_train[i:i+batch_size]

            optimizer.zero_grad()
            logits = model(b_txt, b_asp)
            loss = criterion(logits, b_lbl)
            loss.backward() # Lan truyền ngược (Backpropagation)
            optimizer.step()

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            correct += (preds == b_lbl).sum().item()

        train_acc = correct / len(X_train_txt) * 100

        # Validation trên tập Dev
        model.eval()
        with torch.no_grad():
            dev_logits = model(X_dev_txt, X_dev_asp)
            dev_preds = torch.argmax(dev_logits, dim=1)
            dev_acc = (dev_preds == y_dev).sum().item() / len(X_dev_txt) * 100

        print(f"Epoch {epoch+1:2d} | Train Loss: {total_loss/(len(X_train_txt)/batch_size):.4f} | Train Acc: {train_acc:.2f}% | Dev Acc: {dev_acc:.2f}%")


--- BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH ABSA ---
Epoch  1 | Train Loss: 0.6517 | Train Acc: 74.99% | Dev Acc: 76.65%
Epoch  2 | Train Loss: 0.5765 | Train Acc: 77.26% | Dev Acc: 74.79%
Epoch  3 | Train Loss: 0.5238 | Train Acc: 79.03% | Dev Acc: 75.54%
Epoch  4 | Train Loss: 0.4751 | Train Acc: 80.92% | Dev Acc: 76.27%
Epoch  5 | Train Loss: 0.4298 | Train Acc: 82.78% | Dev Acc: 76.76%
Epoch  6 | Train Loss: 0.3866 | Train Acc: 84.54% | Dev Acc: 77.23%
Epoch  7 | Train Loss: 0.3434 | Train Acc: 86.20% | Dev Acc: 77.64%
Epoch  8 | Train Loss: 0.3006 | Train Acc: 88.17% | Dev Acc: 78.42%
Epoch  9 | Train Loss: 0.2613 | Train Acc: 90.01% | Dev Acc: 78.39%
Epoch 10 | Train Loss: 0.2254 | Train Acc: 91.60% | Dev Acc: 78.01%
Epoch 11 | Train Loss: 0.1942 | Train Acc: 93.02% | Dev Acc: 77.96%
Epoch 12 | Train Loss: 0.1664 | Train Acc: 94.35% | Dev Acc: 77.93%
Epoch 13 | Train Loss: 0.1435 | Train Acc: 95.07% | Dev Acc: 77.49%
Epoch 14 | Train Loss: 0.1256 | Train Acc: 95.94% | Dev Acc: 77.84%
Epoch 1

In [ ]:
# 5. Dự đoán và tổng hợp
def predict_and_vote(sentence_dict, trained_model):
    trained_model.eval()
    text = sentence_dict['text']
    aspects = sentence_dict['aspects']

    if not aspects:
        return text, [], "UNKNOWN"

    # Chuẩn bị dữ liệu text (Batch size = 1)
    tokens = tokenize_vn(text)
    text_idx = [word_vocab.get(w, word_vocab['<UNK>']) for w in tokens[:MAX_SEQ_LEN]]
    text_idx += [word_vocab['<PAD>']] * max(0, MAX_SEQ_LEN - len(text_idx))
    t_text = torch.tensor([text_idx], dtype=torch.long)

    predictions = []
    with torch.no_grad():
        for aspect_name, _ in aspects:
            # Mã hóa Aspect
            asp_idx = aspect_vocab.get(aspect_name, aspect_vocab['<UNK>'])
            t_asp = torch.tensor([asp_idx], dtype=torch.long)

            # Dự đoán
            logits = trained_model(t_text, t_asp)
            pred_class = torch.argmax(logits, dim=1).item()
            predictions.append((aspect_name, inv_polarity_map[pred_class]))

    # Tổng hợp để đưa ra nhãn toàn câu
    pos = sum(1 for _, pol in predictions if pol == 'POSITIVE')
    neg = sum(1 for _, pol in predictions if pol == 'NEGATIVE')

    if pos > neg:     overall = "TÍCH CỰC (POSITIVE)"
    elif neg > pos:   overall = "TIÊU CỰC (NEGATIVE)"
    else:             overall = "TRUNG TÍNH (NEUTRAL)"

    return text, predictions, overall

In [ ]:
# Demo trên tập test
if test_data:
    print("\n--- DEMO DỰ ĐOÁN TRÊN TẬP TEST (Hiển thị 5 câu đầu) ---")
    for i in range(5):
        sample = test_data[i]
        text, asp_preds, overall_sent = predict_and_vote(sample, model)

        print(f"\n[Câu {i+1}]: {text}")
        print("Dự đoán từng khía cạnh:")
        for asp_name, pol in asp_preds:
            print(f"  + {asp_name:20s} ---> {pol}")
        print(f">>> CẢM XÚC TOÀN CÂU: {overall_sent}")
        print("-" * 65)


--- DEMO DỰ ĐOÁN TRÊN TẬP TEST (Hiển thị 5 câu đầu) ---

[Câu 1]: Đây là 1 trong những quán mà mình thích vì vị trà đậm và thơm cũng như mùi vị đặc trưng hơn hẳn những quán khác nè  Trà sữa trân châu sợi - 46k Trà sữa pha khá ngon, vị trà chát và mùi hương khá rõ, không quá ngọt, rất đúng với gu mình  Trà đào - 45k Vị trà đào ở đây cũng đặc biệt hơn hẳn những quán khác, không phải chua ngọt như thưởng thấy mà có mùi trà rất ngon  Cà phê đá xay - 65k Món đá xay ở đây uống cũng ngon không kém trà nè, mùi vị thơm hương cà phê, vị đắng kết hợp hoàn hảo với độ béo ngọt của whipping cream, không quá đắng, cũng không quá ngọt hay lạt lẽo mà dịu nhẹ, thơm và dễ uống lắm  Trà vải thiết quan âm - 45k Trà vải có mùi vị rất thơm ngon mùi vải mà vẫn nghe rõ vị trà, có chút vị chát nhẹ mùi trà thơm rất thích, không phải chỉ toàn vị syrup vải ngọt gắt như nhiều chỗ khác. Do trà ở đây pha khá đậm nên bạn nào uống mà đang đói sẽ dễ say nha, hoặc ban đêm có thể khó ngủ à, cảnh báo trước  Trà thiết quan

## **4. Aspect-Based Sentiment Analysis với VLSP 2018 - Hotel**

- Cuối cùng, bài thực hành demo bài toán: Phân loại cảm xúc cho **từng khía cạnh** (Aspect) xuất hiện trong câu. Sau đó tổng hợp lại để đánh giá cảm xúc của toàn câu.
- **Bộ dữ liệu (Dataset)**: Bộ dữ liệu VLSP 2018 **(Aspect-Based Sentiment Analysis - Hotel)**

In [ ]:
# 1. ĐỌC VÀ TRÍCH XUẤT DỮ LIỆU TỪ FILE TXT VLSP 2018 (Miền: Khách sạn)
polarity_map = {'negative': 0, 'neutral': 1, 'positive': 2}
inv_polarity_map = {0: 'NEGATIVE', 1: 'NEUTRAL', 2: 'POSITIVE'}

def parse_vlsp_absa(filepath):
    if not os.path.exists(filepath):
        print(f"[CẢNH BÁO] Không tìm thấy {filepath}.")
        return []

    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read().strip()

    blocks = re.split(r'\n\s*\n', content)
    parsed_data = []
    aspect_pattern = re.compile(r'\{([^,]+),\s*(positive|negative|neutral)\}')

    for block in blocks:
        lines = block.split('\n')
        if len(lines) >= 3:
            text = lines[1].strip()
            label_line = lines[2].strip()

            aspects = []
            for aspect, polarity in aspect_pattern.findall(label_line):
                aspects.append((aspect.strip(), polarity_map[polarity.strip()]))

            if len(aspects) > 0:
                parsed_data.append({'text': text, 'aspects': aspects})

    return parsed_data

In [ ]:
# Tải dataset từ Google Drive
train_data = parse_vlsp_absa('1-VLSP2018-SA-Hotel-train (7-3-2018).txt')
dev_data = parse_vlsp_absa('2-VLSP2018-SA-Hotel-dev (7-3-2018).txt')
test_data = parse_vlsp_absa('3-VLSP2018-SA-Hotel-test (8-3-2018).txt')

if train_data:
    print(f"\n--- Dữ liệu VLSP 2018 (HOTEL) ---")
    print(f"Số lượng câu - Train: {len(train_data)}, Dev: {len(dev_data)}, Test: {len(test_data)}")
    print(f"Ví dụ câu 1:\n- Text: {train_data[0]['text']}\n- Aspect & Polarity: {train_data[0]['aspects']}")


--- Dữ liệu VLSP 2018 (HOTEL) ---
Số lượng câu - Train: 3000, Dev: 2000, Test: 600
Ví dụ câu 1:
- Text: Rộng rãi KS mới nhưng rất vắng. Các dịch vụ chất lượng chưa cao và thiếu.
- Aspect & Polarity: [('HOTEL#DESIGN&FEATURES', 2), ('HOTEL#GENERAL', 0)]


In [ ]:
# 2. XÂY DỰNG TỪ ĐIỂN (VOCABULARY) CHO TỪ VÀ KHÍA CẠNH KHÁCH SẠN
def tokenize_vn(text):
    return re.findall(r'\w+', text.lower())

word_vocab = {'<PAD>': 0, '<UNK>': 1}
aspect_vocab = {'<UNK>': 0}
word_counter = Counter()

# Đếm tần suất
for item in train_data:
    word_counter.update(tokenize_vn(item['text']))
    for aspect_name, _ in item['aspects']:
        if aspect_name not in aspect_vocab:
            aspect_vocab[aspect_name] = len(aspect_vocab)

# Giới hạn từ điển: 5000 từ phổ biến nhất
for word, _ in word_counter.most_common(5000):
    word_vocab[word] = len(word_vocab)

MAX_SEQ_LEN = 100

# Trải phẳng dữ liệu: 1 câu có N khía cạnh -> N mẫu huấn luyện độc lập
def flatten_and_encode(data):
    X_texts, X_aspects, y_labels = [], [], []
    for item in data:
        tokens = tokenize_vn(item['text'])
        text_idx = [word_vocab.get(w, word_vocab['<UNK>']) for w in tokens[:MAX_SEQ_LEN]]
        text_idx += [word_vocab['<PAD>']] * max(0, MAX_SEQ_LEN - len(text_idx))

        for aspect_name, polarity in item['aspects']:
            aspect_idx = aspect_vocab.get(aspect_name, aspect_vocab['<UNK>'])
            X_texts.append(text_idx)
            X_aspects.append(aspect_idx)
            y_labels.append(polarity)

    return torch.tensor(X_texts, dtype=torch.long), \
           torch.tensor(X_aspects, dtype=torch.long), \
           torch.tensor(y_labels, dtype=torch.long)

X_train_txt, X_train_asp, y_train = flatten_and_encode(train_data)
X_dev_txt, X_dev_asp, y_dev = flatten_and_encode(dev_data)

In [ ]:
# 3. XÂY DỰNG MÔ HÌNH FEEDFORWARD NET CHO ABSA (Dựa trên Hình 6.13)
class ABSAMeanPoolingNet(nn.Module):
    def __init__(self, vocab_size, aspect_size, word_dim, aspect_dim, hidden_dim, num_classes):
        super(ABSAMeanPoolingNet, self).__init__()
        # Lớp nhúng Embeddings (Section 6.5)
        self.word_emb = nn.Embedding(vocab_size, word_dim, padding_idx=0)
        self.aspect_emb = nn.Embedding(aspect_size, aspect_dim)

        # FNN Layers (Section 6.3) - Đầu vào là ghép nối Câu + Khía cạnh
        self.fc1 = nn.Linear(word_dim + aspect_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x_text, x_aspect):
        # 1. Mã hóa câu và Mean-pooling (Eq 6.21: x_mean = 1/N * sum(e_i))
        word_vectors = self.word_emb(x_text)
        sentence_rep = torch.mean(word_vectors, dim=1)

        # 2. Mã hóa khía cạnh
        aspect_rep = self.aspect_emb(x_aspect)

        # 3. Kết hợp (Concatenate)
        combined = torch.cat((sentence_rep, aspect_rep), dim=1)

        # 4. Truyền qua Feedforward Network (Eq 6.22)
        logits = self.fc2(self.relu(self.fc1(combined)))
        return logits

In [ ]:
# Khởi tạo mô hình
model = ABSAMeanPoolingNet(vocab_size=len(word_vocab),
                           aspect_size=len(aspect_vocab),
                           word_dim=128, aspect_dim=32,
                           hidden_dim=64, num_classes=3)

# Sử dụng CrossEntropyLoss (Eq 6.28) tích hợp sẵn Softmax
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.002)

In [ ]:
# 4. HUẤN LUYỆN MÔ HÌNH
epochs = 50
batch_size = 64

if len(X_train_txt) > 0:
    print("\n--- BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH ABSA (HOTEL) ---")
    for epoch in range(epochs):
        model.train()
        total_loss, correct = 0, 0

        for i in range(0, len(X_train_txt), batch_size):
            b_txt = X_train_txt[i:i+batch_size]
            b_asp = X_train_asp[i:i+batch_size]
            b_lbl = y_train[i:i+batch_size]

            optimizer.zero_grad()
            logits = model(b_txt, b_asp)
            loss = criterion(logits, b_lbl)
            loss.backward() # Lan truyền ngược (Backpropagation)
            optimizer.step()

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            correct += (preds == b_lbl).sum().item()

        train_acc = correct / len(X_train_txt) * 100

        # Validation trên tập Dev
        model.eval()
        with torch.no_grad():
            dev_logits = model(X_dev_txt, X_dev_asp)
            dev_preds = torch.argmax(dev_logits, dim=1)
            dev_acc = (dev_preds == y_dev).sum().item() / len(X_dev_txt) * 100

        print(f"Epoch {epoch+1:2d} | Train Loss: {total_loss/(len(X_train_txt)/batch_size):.4f} | Train Acc: {train_acc:.2f}% | Dev Acc: {dev_acc:.2f}%")


--- BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH ABSA (HOTEL) ---
Epoch  1 | Train Loss: 0.5972 | Train Acc: 76.89% | Dev Acc: 77.79%
Epoch  2 | Train Loss: 0.4743 | Train Acc: 82.32% | Dev Acc: 78.75%
Epoch  3 | Train Loss: 0.4313 | Train Acc: 83.81% | Dev Acc: 78.98%
Epoch  4 | Train Loss: 0.3982 | Train Acc: 85.00% | Dev Acc: 79.23%
Epoch  5 | Train Loss: 0.3688 | Train Acc: 86.05% | Dev Acc: 79.44%
Epoch  6 | Train Loss: 0.3415 | Train Acc: 87.12% | Dev Acc: 79.54%
Epoch  7 | Train Loss: 0.3151 | Train Acc: 88.09% | Dev Acc: 79.72%
Epoch  8 | Train Loss: 0.2901 | Train Acc: 89.05% | Dev Acc: 79.23%
Epoch  9 | Train Loss: 0.2661 | Train Acc: 90.12% | Dev Acc: 79.44%
Epoch 10 | Train Loss: 0.2430 | Train Acc: 90.98% | Dev Acc: 79.72%
Epoch 11 | Train Loss: 0.2211 | Train Acc: 91.97% | Dev Acc: 79.90%
Epoch 12 | Train Loss: 0.1987 | Train Acc: 92.68% | Dev Acc: 79.97%
Epoch 13 | Train Loss: 0.1788 | Train Acc: 93.58% | Dev Acc: 80.44%
Epoch 14 | Train Loss: 0.1612 | Train Acc: 94.21% | Dev Acc: 80.42%

In [ ]:
# 5. DỰ ĐOÁN VÀ TỔNG HỢP CẢM XÚC
def predict_and_vote(sentence_dict, trained_model):
    trained_model.eval()
    text = sentence_dict['text']
    aspects = sentence_dict['aspects']

    if not aspects:
        return text, [], "UNKNOWN"

    # Chuẩn bị dữ liệu text (Batch size = 1)
    tokens = tokenize_vn(text)
    text_idx = [word_vocab.get(w, word_vocab['<UNK>']) for w in tokens[:MAX_SEQ_LEN]]
    text_idx += [word_vocab['<PAD>']] * max(0, MAX_SEQ_LEN - len(text_idx))
    t_text = torch.tensor([text_idx], dtype=torch.long)

    predictions = []
    with torch.no_grad():
        for aspect_name, _ in aspects:
            # Mã hóa Aspect
            asp_idx = aspect_vocab.get(aspect_name, aspect_vocab['<UNK>'])
            t_asp = torch.tensor([asp_idx], dtype=torch.long)

            # Dự đoán
            logits = trained_model(t_text, t_asp)
            pred_class = torch.argmax(logits, dim=1).item()
            predictions.append((aspect_name, inv_polarity_map[pred_class]))

    # TỔNG HỢP (VOTING) ĐỂ ĐƯA RA NHÃN TOÀN CÂU
    pos = sum(1 for _, pol in predictions if pol == 'POSITIVE')
    neg = sum(1 for _, pol in predictions if pol == 'NEGATIVE')

    if pos > neg:     overall = "TÍCH CỰC (POSITIVE)"
    elif neg > pos:   overall = "TIÊU CỰC (NEGATIVE)"
    else:             overall = "TRUNG TÍNH (NEUTRAL)"

    return text, predictions, overall

In [ ]:
# DEMO TRÊN TẬP TEST HOTEL
if test_data:
    print("\n--- [PHẦN 3] DEMO DỰ ĐOÁN TRÊN TẬP TEST HOTEL (Hiển thị 5 câu đầu) ---")
    for i in range(5):
        sample = test_data[i]
        text, asp_preds, overall_sent = predict_and_vote(sample, model)

        print(f"\n[Câu {i+1}]: {text}")
        print("Dự đoán từng khía cạnh:")
        for asp_name, pol in asp_preds:
            print(f"  + {asp_name:30s} ---> {pol}")
        print(f">>> CẢM XÚC TOÀN CÂU: {overall_sent}")
        print("-" * 75)


--- [PHẦN 3] DEMO DỰ ĐOÁN TRÊN TẬP TEST HOTEL (Hiển thị 5 câu đầu) ---

[Câu 1]: Ga giường không sạch, nhân viên quên dọn phòng một ngày.
Dự đoán từng khía cạnh:
  + ROOM_AMENITIES#CLEANLINESS     ---> NEGATIVE
  + SERVICE#GENERAL                ---> NEGATIVE
>>> CẢM XÚC TOÀN CÂU: TIÊU CỰC (NEGATIVE)
---------------------------------------------------------------------------

[Câu 2]: Nv nhiệt tình, phòng ở sạch sẽ, tiện nghi, vị trí khá thuận tiện cho việc di chuyển đến các địa điểm ăn + chơi Phòng có gián
Dự đoán từng khía cạnh:
  + SERVICE#GENERAL                ---> POSITIVE
  + ROOMS#CLEANLINESS              ---> POSITIVE
  + ROOMS#COMFORT                  ---> POSITIVE
  + LOCATION#GENERAL               ---> POSITIVE
>>> CẢM XÚC TOÀN CÂU: TÍCH CỰC (POSITIVE)
---------------------------------------------------------------------------

[Câu 3]: Đi bộ ra biển gần, tiện đi lại Phòng view biển nhưng cửa sổ view biển khá bé
Dự đoán từng khía cạnh:
  + LOCATION#GENERAL           